# MuJoCo URDF Import
Onshape can export assemblies in the [Universal Robotic Description Format (URDF)](https://wiki.ros.org/urdf). MuJoCo only has limited support for working with these files directly [(docs section)](https://mujoco.readthedocs.io/en/stable/modeling.html#curdf), so instead parse some key information from it to create a MuJoCo-native MJCF XML file.

## Config

In [1]:
from pathlib import Path

import mujoco as mj

ONSHAPE_ASSEMBLY = "assembly_simplified"

mass_override = {
	"electronics": 0.032
}

# Onshape-exported geometry of wheels is not ideal
geom_override = {
	"wheel": dict(
		type=mj.mjtGeom.mjGEOM_CYLINDER,
		mass=0.01,
		size=[0.0218, 0.0038], # 43.6mm dia, 7.6mm wide
		# mass=0.017,
		# size=[0.03, 0.0038], # 60mm dia, 7.6mm wide
		zaxis=[1, 0, 0],
		pos=[-0.0038, 0, 0]
	),
	"wheel_1": dict(
		type=mj.mjtGeom.mjGEOM_CYLINDER,
		mass=0.01,
		size=[0.0218, 0.0038], # 43.6mm dia, 7.6mm wide
		# mass=0.017,
		# size=[0.03, 0.0038], # 60mm dia, 7.6mm wide
		zaxis=[1, 0, 0],
		pos=[-0.0038, 0, 0]
	),
	"wheel_2": dict(
		type=mj.mjtGeom.mjGEOM_CYLINDER,
		mass=0.01,
		size=[0.0218, 0.0038], # 43.6mm dia, 7.6mm wide
		zaxis=[1, 0, 0],
		pos=[0.0038, 0.0, 0]
	),
	"wheel_3": dict(
		type=mj.mjtGeom.mjGEOM_CYLINDER,
		mass=0.01,
		size=[0.0218, 0.0038], # 43.6mm dia, 7.6mm wide
		zaxis=[1, 0, 0],
		pos=[0.0038, 0.0, 0]
	),
}

cwd = Path(".")
path_assembly = cwd / ONSHAPE_ASSEMBLY
path_urdf = path_assembly / "urdf" / (ONSHAPE_ASSEMBLY + ".urdf")
path_meshes = path_assembly / "meshes"

In [8]:
def mapl(items, func):
	return list(map(func, items))

def parse_vec(string):
	return mapl(string.split(" "), lambda x: float(x))

## Parsing

In [9]:
import xml.etree.ElementTree as ET


xml = ET.parse(path_urdf)

robot = xml.getroot()

links = {}

print("Links (bodies):")
for link in robot.findall("./link"):
	name = link.get("name")
	print(f"  {name}")
	links[name] = {
		"xml": link,
		"children": []
	}

print()
print("Joints:")
for joint in robot.findall("./joint"):
	joint_name = joint.get("name")
	joint_type = joint.get("type")
	if not (joint_type == "fixed" or joint_type == "continuous"):
		print(f"  '{joint_name}' unsupported mate type: {joint_type}")
		continue
	parent = joint.find("./parent").get("link")
	child = joint.find("./child").get("link")
	print(f"  '{joint_name}' {joint_type} {child} to {parent}")
	p = links.get(parent)
	mate = {
		"joint_name": joint_name,
		"joint_type": joint_type,
		"joint_axis": parse_vec(joint.find("./axis").get("xyz")),
		"origin": parse_vec(joint.find("./origin").get("xyz")),
		"child": child
	}
	p["children"].append(mate)


Links (bodies):
  electronics
  lipos
  motor_base
  root
  sensor_imu
  wheel
  wheel_2

Joints:
  'fastened_1' fixed motor_base to root
  'fastened_2' fixed lipos to motor_base
  'fastened_3' fixed electronics to lipos
  'fastened_4' fixed sensor_imu to electronics
  'wheel_left' continuous wheel to motor_base
  'wheel_right' continuous wheel_2 to motor_base


In [11]:
# root = links["root"]
# print(root)
import math

spec = mj.MjSpec.from_file("template.xml")

spec.compiler.meshdir = str(path_meshes)


def convert(mate, parent=spec.worldbody):
	name = mate.get("child")
	link = links.get(name)
	print(name, mate, link)
	body: mj.MjsBody = parent.add_body(name=name, pos=mate["origin"])
	mesh = link["xml"].find("./visual/geometry/mesh")
	color = link["xml"].find("./visual/material/color")
	if color is not None:
		print("  ", "color", color)
		color = parse_vec(color.get("rgba"))
	inertial = link["xml"].find("./inertial")

	if parent is spec.worldbody:
		# root body
		body.add_site(
			name="site_root",
			pos=[0, 0, geom_override["wheel"]["size"][0]],
			size=[0.001, 0.001, 0.001],
		)

	if mesh is not None:
		if inertial is None:
			print(f"warning: mesh {name} has no inertial info")

		mass = float(inertial.find("mass").get("value"))

		mesh_file = Path(mesh.get("filename").replace("package://", "")).relative_to(
			path_meshes
		)
		mesh_name = name
		spec.add_mesh(
			name=mesh_name, file=str(mesh_file), scale=parse_vec(mesh.get("scale"))
		)
		spec.assets[mesh_name] = str(mesh_file)
		geom_name = f"geom_{name}"
		pos = parse_vec(link["xml"].find("./visual/origin").get("xyz"))
		body.add_geom(
			name=geom_name,
			rgba=color,
			**(
				geom_override.get(
					name,
					dict(
						type=mj.mjtGeom.mjGEOM_MESH,
						meshname=mesh_name,
						mass=mass_override.get(name, mass),
						pos=pos,
					),
				)
			),
		)

	else:
		print(f"  {name} has no geometry")

	joint_name = mate["joint_name"]
	match mate["joint_type"]:
		case "freejoint":
			body.add_freejoint(name=joint_name)
		case "fixed":
			pass  # mujoco default
		case "continuous" if not joint_name.startswith("wheel_back"):
			body.add_joint(
				name=f"joint_{joint_name}",
				type=mj.mjtJoint.mjJNT_HINGE,
				axis=mate["joint_axis"],
				# damping=0.001,
				# frictionloss=0.001,
				damping=0,
				frictionloss=0,
				#  armature: motor's rotational inertia reflected through the gearbox
				#            Estimated rotational inertia of N20 motor: 5e-8 Kg-m^2.
				#            With a 1:N gearbox, we multiply the inertia by N^2 to get
				#            the reflected inertia. So, 5e-8 * 30^2 = 4.5e-5 Kg-m^2.
				#            This is ~10x the wheel's inertia, so this dominates. -->
				#
				# armature=4.5e-5,
				armature=0,
			)

			#  Control torque (for a hinge joint) is determined by gear * ctrl (units
			#  of N-m). For reinforcement learning, it helps to have a normalized
			#  ctrl, so we set the limits to [-1, 1]. With the gear set to 0.05, that
			#  gives us actual limits of -0.05 to 0.05 N-m, which is a decent estimate
			#  of our N20 motor with a 1:30 gearbox.

			act = spec.add_actuator(name=f"act_{joint_name}")
			act.damping = [0, 0, 0] # messes with dcmotor..
			act.set_to_dcmotor(
				# measured going ~1m/0.73s, 43.6mm dia wheels, = 600rpm (motor spec 500rpm at 6V, but running at 8V)
				# at 30:1 gearing so shaft = 18000 rpm
				# so motor constant Kv = 0.004244
				# motorconst=[0.004244, 0.004244],
				# motorconst=[0.00382, 0.00382], # 6V, 500rpm
				# motorconst=[0.004, 0.004],
				# motorconst=[0.16, 0.16],
				motorconst=[0, 0],
				# resistance=11.5,  # ohm
				resistance=0,
				nominal=[
					8, 0.15*8/11.5, 8.023*2*math.pi
				],
				saturation=[
					0,
					0.15,
					0,
				],
				inductance=[
					0.0026,  # 2.6mH based on https://www.klikrobotics.com/PWMTutorial2.html
					0,
				],
				thermal=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
				cogging=[0.0, 0.0, 0.0]
			)
			# act.gear[0] = 30 # 30:1 gearbox
			act.ctrlrange = [-8, 8]

			# act.set_to_motor()
			# act.ctrlrange = [-1, 1]
			# # act.gear[0] = 0.05 # original
			# # act.gear[0] = 0.0058 # 1:1
			# # act.gear[0] = 0.007 # fast gearing
			# # act.gear[0] = 0.031 # fastest gearing
			# # new 500rpm motors
			# act.gear[0] = 0.05

			act.target = f"joint_{joint_name}"
			act.trntype = mj.mjtTrn.mjTRN_JOINT

			spec.add_sensor(
				name=f"jointvel_{joint_name}",
				type=mj.mjtSensor.mjSENS_JOINTVEL,
				objtype=mj.mjtObj.mjOBJ_JOINT,
				objname=f"joint_{joint_name}",
			)

	if name.startswith("sensor_"):
		body.add_site(name=f"site_{name}", pos=[0, 0, 0], size=[0.001, 0.001, 0.001])
		sensor_type = name.replace("sensor_", "")
		if sensor_type == "imu":
			# spec.add_sensor(name="imu_accel", type=mj.mjtSensor.mjSENS_ACCELEROMETER, site="site_sensor_imu")
			# spec.add_sensor(name="imu_gyro", type=mj.mjtSensor.mjSENS_GYRO, site="site_sensor_imu")
			# spec.add_sensor(name="imu_orientation", objtype=mj.MjsSite, objname="site_sensor_imu")

			# BNO055 provides orientation quaternion
			spec.add_sensor(
				name="imu_quat",
				type=mj.mjtSensor.mjSENS_FRAMEQUAT,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
				# reftype=mj.mjtObj.mjOBJ_SITE,
				# refname="site_sensor_imu"
			)
			spec.add_sensor(
				name="imu_accel",
				type=mj.mjtSensor.mjSENS_ACCELEROMETER,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)
			spec.add_sensor(
				name="imu_gyro",
				type=mj.mjtSensor.mjSENS_GYRO,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)

	for c in link["children"]:
		convert(c, body)


convert(
	{
		"joint_name": "_root",
		"joint_type": "freejoint",
		# "origin": [0,0,0.5],
		"origin": [0, 0, 0.005],
		"child": "root",
	}
)

# privileged velocity info
spec.add_sensor(
	name="priv_vel",
	type=mj.mjtSensor.mjSENS_VELOCIMETER,
	objtype=mj.mjtObj.mjOBJ_SITE,
	objname="site_root",
)

print()
print(spec.to_xml())

spec.encode("robot.xml")


root {'joint_name': '_root', 'joint_type': 'freejoint', 'origin': [0, 0, 0.005], 'child': 'root'} {'xml': <Element 'link' at 0x72e94f6b43b0>, 'children': [{'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'child': 'motor_base'}]}
  root has no geometry
motor_base {'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'child': 'motor_base'} {'xml': <Element 'link' at 0x72e94f6b4630>, 'children': [{'joint_name': 'fastened_2', 'joint_type': 'fixed', 'joint_axis': [6.93335e-33, -1.35525e-16, -1.0], 'origin': [-2.6001e-18, -4.71628e-18, 0.0401586], 'child': 'lipos'}, {'joint_name': 'wheel_left', 'joint_type': 'continuous', 'joint_axis': [-1.0, -3.20732e-48, -2.31296e-16], 'origin': [-0.02765, -1.30104e-18, 0.0218], 'child': 'wheel'}, {'joint_name': 'wheel_right', 'joint_type': 'continuous', 'joint_axis': [1.0, -3.20732e-48, -2.31296e-16], 'origin': [0.02765, 6.43445e-82, 

3811

# Launch MuJoCo

In [ ]:
import mujoco.viewer

spec = mj.MjSpec.from_file("robot.xml")
model = spec.compile()
data = mj.MjData(model)

mujoco.viewer.launch(model, data)


(python:3704075): dbind-WARNING **: 15:18:51.464: AT-SPI: Error retrieving accessibility bus address: org.freedesktop.DBus.Error.ServiceUnknown: The name is not activatable

(python:3704075): Gtk-WARNING **: 15:18:51.490: Theme parsing error: gtk.css:1048:16: 'border-spacing' is not a valid property name

(python:3704075): Gtk-WARNING **: 15:18:51.526: Theme parsing error: gtk.css:1048:16: 'border-spacing' is not a valid property name
/nix/store/8w97xwciwfg8zs6yp7a1dfyflz9y2slh-python3-3.13.15-env/lib/python3.13/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


: 

In [60]:
import time

import mujoco as mj
import mujoco.viewer as mjv

from types import SimpleNamespace
from scipy.spatial.transform import Rotation
import numpy as np

spec = mj.MjSpec.from_file("robot.xml")
model = spec.compile()
data = mj.MjData(model)

try:
	viewer.close()
except:
	pass

step = 0

a1 = data.actuator("act_wheel_left")
a2 = data.actuator("act_wheel_right")
jv1 = data.sensor("jointvel_wheel_left")
robot = data.body("motor_base")

d = SimpleNamespace(
			robot=data.body("root"),  # to get position
			act1=data.actuator("act_wheel_left"),
			act2=data.actuator("act_wheel_right"),
			imu_gyro=data.sensor("imu_gyro"),
			imu_quat=data.sensor("imu_quat"),
			priv_vel=data.sensor("priv_vel"),
			input="voltage"
		)


a1.ctrl = -8
a2.ctrl = 8

mj.mj_forward(model, data)

print("robot mass", model.body("root").subtreemass)
print("robot center of mass", robot.subtree_com)
print("axle position", data.body("wheel").xpos[2])
print("pendulum arm", robot.subtree_com[2] - data.body("wheel").xpos[2])
vels = []

viewer = mjv.launch_passive(model, data)

while True:
	step += 1
	step_start = time.time()
	mj.mj_step(model, data)
	viewer.sync()

	gyro = d.imu_gyro.data
	quat = d.imu_quat.data

	r = Rotation.from_quat(quat, scalar_first=True)
	r.inv()

	eul = r.as_euler("xyz", degrees=False, suppress_warnings=True)
	if step % 30 == 0:
		print(step, "vel", jv1.data[0])
		# print(step, "eul", eul)
		# print(" " * (len(f"{step}")), "vel", d.priv_vel.data)


	# if step == 400:
	# 	pos = robot.xpos[1]
	# if step > 400:
	# 	vels.append(jv1.data[0])
	# if step > 800:
	# 	print("moved", (robot.xpos[1] - pos)/2, "m/s")
	# 	print("avg vel steps 400-800", np.mean(vels)/(2*np.pi), "rotations per second")
	# 	viewer.close()
	# 	time.sleep(1)
	# 	break

	# print(step, j1.qvel, j1.qacc)

	sleep = 0.005 - (time.time() - step_start)
	if sleep > 0:
		time.sleep(sleep)


robot mass [0.1368748]
robot center of mass [2.60617723e-10 7.86753385e-11 4.65098437e-02]
axle position 0.0268
pendulum arm 0.01970984366195787
30 vel -50.409895462479284
60 vel -50.409895462479284
90 vel -50.409895462479284
120 vel -50.409895462479284
150 vel -50.409895462479284
180 vel -50.409895462479284
210 vel -50.409895462479284
240 vel -50.409895462479284
270 vel -50.409895462479284
300 vel -50.409895462479284
330 vel -50.409895462479284
360 vel -50.409895462479284
390 vel -50.409895462479284
420 vel -50.409895462479284
450 vel -50.409895462479284
480 vel -50.409895462479284
510 vel -50.409895462479284
540 vel -50.409895462479284
570 vel -50.409895462479284
600 vel -50.409895462479284
630 vel -50.409895462479284
660 vel -50.409895462479284
690 vel -50.409895462479284
720 vel -50.409895462479284
750 vel -50.409895462479284
780 vel -50.409895462479284
810 vel -50.409895462479284
840 vel -50.409895462479284
870 vel -50.409895462479284
900 vel -50.409895462479284
930 vel -50.409895

KeyboardInterrupt: 

In [ ]:
viewer.close()

: 

: 

: 

: 